# Choose kernels

A kernel updates one block of parameters conditional on the current values of
the other blocks. Goose applies the kernels in sequence within each iteration.
You can use different methods for different blocks without changing the model's
priors or likelihood.

In [1]:
import jax.numpy as jnp
import numpy as np
import pandas as pd
import tensorflow_probability.substrates.jax.bijectors as tfb
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel.goose as gs
import liesel.model as lsl

## Prepare the example

These examples require the regression `model`, including its transformation
and inference specifications, from {doc}`tutorials/md/01c-transform`.

In [2]:
# Python cell fragment: imports are visible in each guide.
# Simulate regression observations.
rng = np.random.default_rng(42)
n = 500
true_beta = np.array([1.0, 2.0])
true_sigma = 1.0
x = rng.uniform(size=n)
X_mat = np.column_stack([np.ones(n), x])
y_vec = X_mat @ true_beta + rng.normal(scale=true_sigma, size=n)

# Define the priors.
beta = lsl.Var.new_param(
    jnp.zeros(2),
    dist=lsl.Dist(tfd.Normal, 0.0, 5.0),
    name="beta",
)
sigma_sq = lsl.Var.new_param(
    1.0,
    dist=lsl.Dist(tfd.InverseGamma, concentration=3.0, scale=2.0),
    name="sigma_sq",
)

# Connect the parameters to the observations.
sigma = lsl.Var.new_calc(jnp.sqrt, sigma_sq, name="sigma")
X = lsl.Var.new_obs(X_mat, name="X")
mu = lsl.Var.new_calc(jnp.dot, X, beta, name="mu")
y = lsl.Var.new_obs(
    y_vec,
    dist=lsl.Dist(tfd.Normal, mu, sigma),
    name="y",
)

sigma_sq.biject(tfb.Exp(), name="log_sigma_sq")
log_sigma_sq = sigma_sq.bijected_var

joint = gs.MCMCSpec(
    gs.NUTSKernel,
    kernel_group="regression",
    jitter_dist=tfd.Normal(0.0, 0.2),
)
beta.inference = joint
log_sigma_sq.inference = joint
model = lsl.Model(y)


## Choose an update

| Kernel | Use it for | What you supply or check |
| --- | --- | --- |
| {class}`~liesel.goose.NUTSKernel` | Continuous parameters with gradients. | Parameters on an unconstrained scale; trajectory length is selected automatically. |
| {class}`~liesel.goose.HMCKernel` | Hamiltonian updates with a chosen trajectory length. | Gradients and the number of integration steps. |
| {class}`~liesel.goose.IWLSKernel` | Blocks where local curvature gives a useful Gaussian proposal. | A differentiable density with suitable curvature; larger blocks make matrix operations more expensive. |
| {class}`~liesel.goose.RWKernel` | A simple Gaussian random-walk update. | A useful parameter scale; mixing can be slow for strongly dependent parameters. |
| {class}`~liesel.goose.GibbsKernel` | A block whose full conditional you can sample directly. | A function drawing from that full conditional. |
| {class}`~liesel.goose.MHKernel` | A custom Metropolis-Hastings proposal. | A proposal function and the backward-minus-forward log proposal correction. |

See {doc}`tutorials/md/01c-transform` for a positive parameter sampled on a
log scale, and {doc}`tutorials/md/01d-gibbs-sampling` for an exact Gibbs update.
For custom proposals, start with {doc}`tutorials/md/08-custom-kernel`.

## Check kernel assignments

For the regression `model` from {doc}`tutorials/md/01c-transform`:

In [3]:
mcmc = gs.LieselMCMC(model)
kernel_table = pd.DataFrame(
    [
        {"Kernel": type(kernel).__name__, "Parameters": kernel.position_keys}
        for kernel in mcmc.get_kernel_list()
    ]
)

In [4]:
kernel_table

,Kernel,Parameters
0,NUTSKernel,"(beta, log_sigma_sq)"


Check that every parameter you intend to estimate appears in a kernel.
A parameter without an inference specification stays at its current value
unless you add a kernel manually through an {doc}`engine builder <goose-engine>`.
Declaring a prior and setting the parameter flag does not assign a kernel. If
you sample regression coefficients but leave the variance without a kernel,
you obtain a posterior conditional on that fixed variance, not uncertainty
about both. See {doc}`model-modification` for deliberate model changes.

## Group parameters

For the `model` from {doc}`tutorials/md/01c-transform`, give the coefficients
and log variance separate NUTS kernels:

In [5]:
beta = model.vars["beta"]
log_variance = model.vars["log_sigma_sq"]

beta.inference = gs.MCMCSpec(gs.NUTSKernel, order=1)
log_variance.inference = gs.MCMCSpec(gs.NUTSKernel, order=2)

One iteration first updates `beta` given the old log variance, then updates
the log variance given the new `beta`. The vector `beta` is already one
block: both coefficients are proposed together.

To update both variables jointly, give them an explicit shared group:

In [6]:
joint = gs.MCMCSpec(
    gs.NUTSKernel,
    kernel_group="regression",
    kernel_kwargs={"da_target_accept": 0.9},
    jitter_dist=tfd.Normal(0.0, 0.2),
)
beta.inference = joint
log_variance.inference = joint

The shared specification keeps the group's kernel, arguments, and order
consistent. Reusing a specification without `kernel_group` does **not**
combine variables. Assigning a new specification replaces the previous one,
including its jitter settings.

Joint updates can help with posterior dependence but cost more per update.
Inspect {doc}`diagnostics <goose-diagnostics>` after changing the grouping.
A higher `da_target_accept` generally adapts toward smaller steps; it is not
a substitute for a well-scaled model.

## Order the updates

Smaller `order` values run first. With equal values, the high-level helper
uses reverse graph order, starting nearer the responses. Inspect
{meth}`get_kernel_list() <liesel.goose.LieselMCMC.get_kernel_list>` if the order matters, and use the same order for every
member of a joint group. See {class}`~liesel.goose.MCMCSpec` and
{meth}`~liesel.goose.LieselMCMC.get_kernel_groups` for the exact grouping rules.

<figure aria-label="Two sequential parameter updates in one iteration">
  <div style="display:flex;flex-wrap:wrap;gap:.6rem;align-items:center;padding:1rem;border:1px solid #999;border-radius:.4rem">
    <span>Current β and σ²</span><span aria-hidden="true">→</span>
    <strong>Update β given σ²</strong><span aria-hidden="true">→</span>
    <strong>Update σ² given new β</strong><span aria-hidden="true">→</span>
    <span>Store the iteration</span>
  </div>
  <figcaption>Each block sees updates already made in this iteration. Separate chains run their own sequences.</figcaption>
</figure>